In [ ]:
%run kaggle/setup.py

In [ ]:
# Verify cache exists
import os
cache_path = '/kaggle/working/cache/index.json'
if os.path.exists(cache_path):
    import json
    with open(cache_path) as f:
        index = json.load(f)
    print(f"Cache found: {cache_path}")
    for split in ['train', 'val', 'test']:
        print(f"  {split}: {len(index.get(split, []))} clips")
else:
    raise FileNotFoundError("Cache not found! Run preprocess.ipynb first.")

In [ ]:
# Phase 1: Audio-only training (200K steps)
import subprocess
import sys

cmd = [
    sys.executable, 'scripts/train.py',
    'train=phase1',
    'train.max_steps=200000',
    'train.checkpoint_every_n_steps=5000',
    'data.index_file=/kaggle/working/cache/index.json',
    'data.num_workers=2',
    'train.precision=16-mixed',
    'logger.save_dir=/kaggle/working/logs',
]
print(f"Running Phase 1: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    raise RuntimeError(f"Phase 1 training failed with code {result.returncode}")

In [ ]:
# Phase 2: Cross-modal attention warmup (30K steps)
import subprocess
import sys
import glob

# Find best Phase 1 checkpoint
phase1_checkpoints = glob.glob('/kaggle/working/checkpoints/phase1/*.ckpt')
if phase1_checkpoints:
    phase1_best = max(phase1_checkpoints, key=os.path.getctime)
    print(f"Loading from: {phase1_best}")
else:
    # Fallback to last.ckpt
    phase1_best = '/kaggle/working/checkpoints/phase1/last.ckpt'

cmd = [
    sys.executable, 'scripts/train.py',
    'train=phase2',
    'train.max_steps=30000',
    'train.checkpoint_every_n_steps=5000',
    f'train.resume_from_checkpoint={phase1_best}',
    'data.index_file=/kaggle/working/cache/index.json',
    'data.num_workers=2',
    'train.precision=16-mixed',
    'logger.save_dir=/kaggle/working/logs',
]
print(f"Running Phase 2: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    raise RuntimeError(f"Phase 2 training failed with code {result.returncode}")

In [ ]:
# Phase 3 Config A: End-to-end fine-tuning with DINOv2 frozen (100K steps)
import subprocess
import sys
import glob
import os

# Find best Phase 2 checkpoint
phase2_checkpoints = glob.glob('/kaggle/working/checkpoints/phase2/*.ckpt')
if phase2_checkpoints:
    phase2_best = max(phase2_checkpoints, key=os.path.getctime)
    print(f"Loading from: {phase2_best}")
else:
    phase2_best = '/kaggle/working/checkpoints/phase2/last.ckpt'

cmd = [
    sys.executable, 'scripts/train.py',
    'train=phase3_config_a',
    'train.max_steps=100000',
    'train.checkpoint_every_n_steps=5000',
    f'train.resume_from_checkpoint={phase2_best}',
    'data.index_file=/kaggle/working/cache/index.json',
    'data.num_workers=2',
    'train.precision=16-mixed',
    'logger.save_dir=/kaggle/working/logs',
]
print(f"Running Phase 3 Config A: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    raise RuntimeError(f"Phase 3 training failed with code {result.returncode}")

In [ ]:
# Run all evaluation scripts
import subprocess
import sys
import glob
import os

# Find best Phase 3 checkpoint
phase3_checkpoints = glob.glob('/kaggle/working/checkpoints/phase3/*.ckpt')
if phase3_checkpoints:
    phase3_best = max(phase3_checkpoints, key=os.path.getctime)
    print(f"Loading from: {phase3_best}")
else:
    phase3_best = '/kaggle/working/checkpoints/phase3/last.ckpt'

checkpoint = phase3_best
index_file = '/kaggle/working/cache/index.json'

# SI-SNRi with PIT
print("Running SI-SNRi evaluation...")
subprocess.run([sys.executable, 'evaluation/eval_sisnri.py',
    '--checkpoint', checkpoint,
    '--index_file', index_file,
    '--output', '/kaggle/working/outputs/eval_sisnri.json'])

# SDR/SIR/SAR
print("Running SDR/SIR/SAR evaluation...")
subprocess.run([sys.executable, 'evaluation/eval_sdr.py',
    '--checkpoint', checkpoint,
    '--index_file', index_file,
    '--output', '/kaggle/working/outputs/eval_sdr.json'])

# Localisation (IoU) with top-50 patches
print("Running localisation evaluation...")
subprocess.run([sys.executable, 'evaluation/eval_localisation.py',
    '--checkpoint', checkpoint,
    '--index_file', index_file,
    '--output', '/kaggle/working/outputs/eval_localisation.json',
    '--use_yolo'])

# WER with baseline
print("Running WER evaluation...")
subprocess.run([sys.executable, 'evaluation/eval_wer.py',
    '--checkpoint', checkpoint,
    '--index_file', index_file,
    '--output', '/kaggle/working/outputs/eval_wer.json'])

# Zero-shot
print("Running zero-shot evaluation...")
subprocess.run([sys.executable, 'evaluation/eval_zero_shot.py',
    '--checkpoint', checkpoint,
    '--index_file', index_file,
    '--output', '/kaggle/working/outputs/eval_zero_shot.json'])

print("\nAll evaluations complete!")

In [ ]:
# Print results table and display attention visualisation
import json
import matplotlib.pyplot as plt
import numpy as np

print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

# Load all results
results_files = [
    ('SI-SNRi (PIT)', '/kaggle/working/outputs/eval_sisnri.json'),
    ('SDR/SIR/SAR', '/kaggle/working/outputs/eval_sdr.json'),
    ('Localisation', '/kaggle/working/outputs/eval_localisation.json'),
    ('WER', '/kaggle/working/outputs/eval_wer.json'),
    ('Zero-shot', '/kaggle/working/outputs/eval_zero_shot.json'),
]

for name, path in results_files:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f"\n{name}:")
        for k, v in data.items():
            if k not in ['per_sample', 'per_source', 'seen_categories', 'unseen_categories', 'seen_clips', 'unseen_clips']:
                print(f"  {k}: {v}")
    else:
        print(f"\n{name}: NOT FOUND at {path}")

# Display attention visualisation if localisation results exist
loc_path = '/kaggle/working/outputs/eval_localisation.json'
if os.path.exists(loc_path):
    with open(loc_path) as f:
        loc_data = json.load(f)
    
    print(f"\nLocalisation Details:")
    print(f"  Loc Acc (IoU>0.3): {loc_data.get('loc_acc', 0):.3f}")
    print(f"  Mean IoU: {loc_data.get('mean_iou', 0):.3f}")
    print(f"  Frames: {loc_data.get('n_frames_evaluated', 0)}")
    
    # Plot per-sample IoU if available
    if 'per_sample_iou' in loc_data:
        plt.figure(figsize=(10, 4))
        plt.plot(loc_data['per_sample_iou'], 'o-', markersize=3)
        plt.axhline(y=0.3, color='r', linestyle='--', label='IoU=0.3 threshold')
        plt.xlabel('Frame Index')
        plt.ylabel('IoU')
        plt.title('Per-Frame IoU (Top-50 Patches vs YOLOv8 GT)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('/kaggle/working/outputs/localisation_iou.png', dpi=150)
        plt.show()
        print("\nAttention visualisation saved to /kaggle/working/outputs/localisation_iou.png")